In [23]:
import numpy as np
import pandas as pd


#1. ormalize
def minmax_normalize(series):
    s = series.astype(float).copy()
    s_min = s.min()
    s_max = s.max()

    if pd.isna(s_min) or pd.isna(s_max):
        return pd.Series(np.zeros(len(s)), index=s.index)

    if s_max == s_min:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s - s_min) / (s_max - s_min)


#2. build item-level signals from synthetic history
def build_item_signal_table(
    synthetic_view_history,
    item_col="title",
    save_col="save",
    listen_ratio_col="listen_ratio"
):
    df = synthetic_view_history.copy()

    df["save_binary"] = (df[save_col] == "yes").astype(int)
    df[listen_ratio_col] = pd.to_numeric(df[listen_ratio_col], errors="coerce")

    item_stats = (
        df.groupby(item_col, as_index=False)
          .agg(
              plays=(item_col, "size"),
              avg_listen_ratio=(listen_ratio_col, "mean"),
              save_rate=("save_binary", "mean")
          )
    )

    return item_stats


#3. compute exposure / quality / inclusion signals
def compute_exposure_fairness_signals(
    item_stats,
    item_col="title",
    alpha=0.7
):
    """
    alpha = weight for listen ratio
    (1 - alpha) = weight for save rate
    """
    df = item_stats.copy()

    # Exposure: normalized play count
    df["exposure_score"] = minmax_normalize(df["plays"])

    # Normalize engagement signals
    df["listen_time_score"] = minmax_normalize(df["avg_listen_ratio"])
    df["save_score"] = minmax_normalize(df["save_rate"])

    # Simplified quality proxy
    df["quality_score"] = alpha * df["listen_time_score"] + (1 - alpha) * df["save_score"]

    # Inclusion signal
    df["inclusion_score"] = (1 - df["exposure_score"]) * df["quality_score"]

    return df[[item_col, "plays", "avg_listen_ratio", "save_rate",
               "exposure_score", "listen_time_score", "save_score", "quality_score", "inclusion_score"]]


#4. rerank baseline recommendations
def rerank_with_exposure_fairness(
    baseline_recs,
    item_signal_table,
    user_col="user_id",
    item_col="title",
    score_col="baseline_score",
    lambda_=0.6,
    theta_r=0.3,
    top_k=10
):
    """
    baseline_recs expected columns:
    - user_id
    - title (or item key)
    - baseline_score
    """
    recs = baseline_recs.copy()
    signals = item_signal_table.copy()

    # merge item-level fairness signals onto baseline recs
    merged = recs.merge(signals, on=item_col, how="left")

    # if some items have no signal yet, fill with 0
    for col in ["exposure_score", "listen_time_score", "save_score", "quality_score", "inclusion_score"]:
        if col in merged.columns:
            merged[col] = merged[col].fillna(0)

    # relevance safeguard
    merged["eligible"] = merged[score_col] >= theta_r

    # final score
    merged["final_score"] = np.where(
        merged["eligible"],
        lambda_ * merged[score_col] + (1 - lambda_) * merged["inclusion_score"],
        merged[score_col]
    )

    # sort within each user
    merged = merged.sort_values(
        by=[user_col, "final_score"],
        ascending=[True, False]
    )

    # assign reranked position
    merged["rerank_position"] = merged.groupby(user_col).cumcount() + 1

    if top_k is not None:
        merged = merged[merged["rerank_position"] <= top_k].copy()

    return merged

In [24]:
import pandas as pd

synthetic_view_history = pd.read_csv(
    "/Users/tricialee/Documents/MasterUU/Media/ASM/PreferenceSystem/INFOMPPM/synthetic_view_history_simple.csv"
)

item_stats = build_item_signal_table(
    synthetic_view_history,
    item_col="title",
    save_col="save",
    listen_ratio_col="listen_ratio"
)

item_signal_table = compute_exposure_fairness_signals(
    item_stats,
    item_col="title",
    alpha=0.7
)

item_signal_table.head()

,title,plays,avg_listen_ratio,save_rate,exposure_score,listen_time_score,save_score,quality_score,inclusion_score
0,#CancelKarenDunbar,61,0.347049,0.098361,0.009098,0.224356,0.098361,0.186557,0.184860
1,'Go Back To Where You Came From',46,0.298913,0.021739,0.006823,0.155590,0.021739,0.115435,0.114647
2,"'Til Kingdom Come: Trump, Faith and Money",908,0.352081,0.055066,0.137528,0.231545,0.055066,0.178601,0.154039
3,"1Xtra - 1XTRA Talks: Pain, Power and Progress",393,0.814122,0.251908,0.059439,0.891603,0.251908,0.699695,0.658106
4,1Xtra - 1Xtra's Hot 4 2022,176,0.347443,0.039773,0.026535,0.224919,0.039773,0.169375,0.164881


In [27]:
bbc_seen = pd.read_csv('bbc_seen_items.csv')
bbc_seen = bbc_seen.merge(
    item_signal_table[["title", "inclusion_score"]],
    on="title",
    how="left"
)

bbc_seen.head()

,category,title,tags,description,image,duration_txt,duration_sec,first_broadcast,synopsis_small,synopsis_medium,synopsis_large,popularity_group,popularity_rank,quality_label,view_count,I_i,inclusion_score
0,science-and-nature,Arctic with Bruce Parry - 2. Greenland,"BBC, iPlayer, TV, Arctic with Bruce Parry, 2. ...",Bruce Parry meets the last traditional Inuit h...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,59 mins,3549.0,9pm 9 Jan 2011,Bruce Parry meets the last traditional Inuit h...,Bruce Parry journeys to the far north of Green...,Bruce Parry journeys to the far north of Green...,head,1,bad,496,0.161623,0.161623
1,science-and-nature,Life of a Mountain - A Year on Helvellyn,"BBC, iPlayer, TV, Life of a Mountain, A Year o...",Terry Abrahams film of Helvellyn features a lo...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,89 mins,5326.0,26 Jan 2021,Terry Abraham's film of Helvellyn features a l...,Terry Abraham films a year in the life of Helv...,This spectacular film features a year in the l...,head,2,good,504,0.655867,0.655867
2,science-and-nature,Click - Short Edition - 04/12/2021,"BBC, iPlayer, TV, Click - Short Edition, 04/12...","A guide to all the latest gadgets, websites, g...",https://ichef.bbci.co.uk/images/ic/1200x675/p0...,12 mins,715.0,3:30am 4 Dec 2021,"A guide to all the latest gadgets, websites, g...",A comprehensive guide to all the latest gadget...,A comprehensive guide to all the latest gadget...,head,3,bad,453,0.172002,0.172002
3,science-and-nature,The Computer Programme - 1. It's Happening Now,"BBC, iPlayer, TV, The Computer Programme, 1. I...","Who is using computers now, and where is this ...",https://ichef.bbci.co.uk/images/ic/1200x675/p0...,25 mins,1486.0,11 Jan 1982,"Who is using computers now, and where is this ...",The Computer Programme asks: what can computer...,"Chris Serle, Ian McNaught-Davis and Gill Nevil...",head,4,good,434,0.652168,0.652168
4,science-and-nature,Serengeti II - Series 1: 5. Power,"BBC, iPlayer, TV, Serengeti II, Series 1: 5. P...",Power is shifting among the families as their ...,https://ichef.bbci.co.uk/images/ic/1200x675/p0...,58 mins,3480.0,22 Aug 2021,Power is shifting among the families as their ...,Power is shifting among the animal families as...,"For Kali, the threat from Sefu has been replac...",head,5,good,489,0.589253,0.589253
